## Training notebook

In [1]:
import gym
import highway_env

from stable_baselines3 import PPO
#from d3rlpy.algos import DiscreteSAC
#from d3rlpy.online.buffers import ReplayBuffer

%load_ext tensorboard
from datetime import datetime
import os
from ipywidgets import interact, widgets
from stable_baselines3.common.callbacks import CheckpointCallback, EvalCallback, BaseCallback

from typing import Callable

### Select operating system

In [2]:
os_id = widgets.Text()
def f(desired_os):
    os_id.value = str(desired_os)
interact(f, desired_os=['UBUNTU_PIGO', 'MACOS', 'WINDOWS', 'UBUNTU_MICRO', 'UBUNTU_DELL'])

interactive(children=(Dropdown(description='desired_os', options=('UBUNTU_PIGO', 'MACOS', 'WINDOWS', 'UBUNTU_M…

<function __main__.f(desired_os)>

In [3]:
OUTPUT_PATH = '/Users/fornerispighetti/HighwayDRL/training_output/' if os_id.value == "MACOS" else 'C:/Users/pigo/Desktop/HighwayDRL/training_output/'
if os_id.value == 'UBUNTU_MICRO':
    OUTPUT_PATH = '/home/utente1/Desktop/PighettiForneris/HighwayDRL/training_output/'
elif os_id.value == 'UBUNTU_DELL':
    OUTPUT_PATH = '/home/utente1/pighetti/HighwayDRL/training_output'
elif os_id.value == 'UBUNTU_PIGO':
    OUTPUT_PATH = '/home/pigo/projects/HighwayDRL/training_output'

### Select environment

In [4]:
env_id = widgets.Text()
def f(input_env):
    env_id.value = str(input_env)
interact(f, input_env=['highway-v0','dm-env-v0','acc-d<m-env-v0','jam-dm-env-v0','overtaken-dm-env-v0','singleO-dm-env-v0','multipleO-dm-env-v0'])

interactive(children=(Dropdown(description='input_env', options=('highway-v0', 'dm-env-v0', 'acc-d<m-env-v0', …

<function __main__.f(input_env)>

In [5]:
total_timesteps = 1e6
# Create and wrap the environment
env = gym.make(env_id.value)
env.configure({
    "training_total_timesteps": total_timesteps
})
# Create a TensorboardCallback to plot sub-rewards

class TensorboardCallback(BaseCallback):
    def __init__(self, verbose=1):
        super(TensorboardCallback, self).__init__(verbose)
        self.coll_rew = 0
        self.rml_rew = 0
        self.high_speed_rew = 0
        self.current_steps = 0

    def _on_step(self) -> bool:        
        self.rml_rew += self.training_env.get_attr('rml_reward')[0]
        self.high_speed_rew += self.training_env.get_attr('high_speed_reward')[0]
            
        if self.locals["dones"][0]:
            self.coll_rew += -self.training_env.get_attr('collision_reward')[0] * self.training_env.get_attr('vehicle')[0].crashed
            self.logger.record("eval/coll_rew", self.coll_rew)
            self.logger.record("episode/RML_rew", self.rml_rew / self.current_steps)
            self.logger.record("episode/high_speed_rew", self.high_speed_rew / self.current_steps)
            self.rml_rew = self.high_speed_rew = self.current_steps = 0
        else:
            self.current_steps = self.training_env.get_attr('steps')[0]
        return True

# Save a checkpoint every 5000 steps
checkpoint_callback = CheckpointCallback(save_freq=5000, save_path=f'{OUTPUT_PATH}/checkpoints',
                                         name_prefix='dm_rl_callback')

eval_callback = EvalCallback(env, best_model_save_path=f'{OUTPUT_PATH}/eval_logs',
                             log_path=f'{OUTPUT_PATH}/eval_logs', eval_freq=1000,
                             deterministic=True, render=False)

In [6]:
def linear_schedule(initial_value: float) -> Callable[[float], float]:
    """
    Linear learning rate schedule.

    :param initial_value: Initial learning rate.
    :return: schedule that computes
      current learning rate depending on remaining progress
    """
    def func(progress_remaining: float) -> float:
        """
        Progress will decrease from 1 (beginning) to 0.

        :param progress_remaining:
        :return: current learning rate
        """
        return progress_remaining * initial_value

    return func

In [7]:
ppo_ilr = 3.5e-4
# PPO parameters
model = PPO("MlpPolicy",
        env,
        gamma=0.97,
        batch_size=512,
        n_steps=16384,
        n_epochs=15,
        ent_coef=0.1,
        learning_rate=ppo_ilr,
        verbose=1,
        tensorboard_log=f'{OUTPUT_PATH}/logdir'
        )

Using cuda device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


In [8]:
try:
    # Train the agent for n steps
    model.learn(total_timesteps=int(total_timesteps), callback=[checkpoint_callback, eval_callback, TensorboardCallback()])
finally:
    # Save the trained agent
    model.save(f'{OUTPUT_PATH}/models/ppo_NEWTRAIN_{str(os_id.value)}_{total_timesteps:.1E}_{env_id.value}_{datetime.now().strftime("%d-%m_%H-%M-%S")}.zip')

Logging to /home/pigo/projects/HighwayDRL/training_output/logdir/PPO_15


/home/pigo/miniconda3/envs/highway/lib/python3.8/site-packages/stable_baselines3/common/evaluation.py:67: UserWarning: Evaluation environment is not wrapped with a ``Monitor`` wrapper. This may result in reporting modified episode lengths and rewards, if other wrappers happen to modify these. Consider wrapping environment first with ``Monitor`` wrapper.
  warnings.warn(


Eval num_timesteps=1000, episode_reward=0.19 +/- 2.39
Episode length: 34.60 +/- 42.82
---------------------------------
| episode/           |          |
|    RML_rew         | 0.1      |
|    high_speed_rew  | 0.0766   |
| eval/              |          |
|    coll_rew        | 0        |
|    mean_ep_length  | 34.6     |
|    mean_reward     | 0.195    |
| time/              |          |
|    total_timesteps | 1000     |
---------------------------------
New best mean reward!
Eval num_timesteps=2000, episode_reward=-1.00 +/- 0.00
Episode length: 15.60 +/- 5.85
---------------------------------
| episode/           |          |
|    RML_rew         | 0.176    |
|    high_speed_rew  | 0.183    |
| eval/              |          |
|    coll_rew        | 0        |
|    mean_ep_length  | 15.6     |
|    mean_reward     | -1       |
| time/              |          |
|    total_timesteps | 2000     |
---------------------------------
Eval num_timesteps=3000, episode_reward=2.74 +/- 3.05
Epis

KeyboardInterrupt: 

## Continued learning

In [5]:
env_cl_id = 'dm-env-v0'
env_cl = gym.make(env_cl_id)


OUTPUT_PATH = '/home/pigo/projects/HighwayDRL/training_output/'

In [6]:
custom_objects = { 'learning_rate': 3.5e-4 }

model_cl = PPO.load(f"{OUTPUT_PATH}models/ppo_NEWTRAIN_cl2_6.5E+05_dm-env-v0_10-03_17-05-10", env=env_cl, tensorboard_log=f"{OUTPUT_PATH}logdir")

Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


In [7]:
class TensorboardCallback(BaseCallback):
    def __init__(self, verbose=1):
        super(TensorboardCallback, self).__init__(verbose)
        self.coll_rew = 0
        self.rml_rew = 0
        self.high_speed_rew = 0
        self.current_steps = 0

    def _on_step(self) -> bool:        
        self.rml_rew += self.training_env.get_attr('rml_reward')[0]
        self.high_speed_rew += self.training_env.get_attr('high_speed_reward')[0]
            
        if self.locals["dones"][0]:
            self.coll_rew += -self.training_env.get_attr('collision_reward')[0]
            self.logger.record("eval/coll_rew", self.coll_rew)
            self.logger.record("episode/RML_rew", self.rml_rew / self.current_steps)
            self.logger.record("episode/high_speed_rew", self.high_speed_rew / self.current_steps)
            self.rml_rew = self.high_speed_rew = self.current_steps = 0
        else:
            self.current_steps = self.training_env.get_attr('steps')[0]
        return True

# Save a checkpoint every 1000 steps
checkpoint_callback = CheckpointCallback(save_freq=5000, save_path=f'{OUTPUT_PATH}checkpoints',
                                         name_prefix='dm_rl_callback')

eval_callback = EvalCallback(env_cl, best_model_save_path=f'{OUTPUT_PATH}/eval_logs',
                             log_path=f'{OUTPUT_PATH}/eval_logs', eval_freq=1000,
                             deterministic=True, render=False)

In [8]:
new_timesteps = 3e5
try:
    model_cl.learn(total_timesteps=int(new_timesteps), callback=[checkpoint_callback, eval_callback, TensorboardCallback()], reset_num_timesteps=False)
finally:
    model_cl.save(f'{OUTPUT_PATH}models/ppo_NEWTRAIN_cl3_aggressive_{env_cl_id}_{datetime.now().strftime("%d-%m_%H-%M-%S")}.zip')

Logging to /home/pigo/projects/HighwayDRL/training_output/logdir/PPO_15


/home/pigo/miniconda3/envs/highway/lib/python3.8/site-packages/stable_baselines3/common/evaluation.py:67: UserWarning: Evaluation environment is not wrapped with a ``Monitor`` wrapper. This may result in reporting modified episode lengths and rewards, if other wrappers happen to modify these. Consider wrapping environment first with ``Monitor`` wrapper.
  warnings.warn(


Eval num_timesteps=686379, episode_reward=30.98 +/- 5.86
Episode length: 120.00 +/- 0.00
---------------------------------
| episode/           |          |
|    RML_rew         | 0.0223   |
|    high_speed_rew  | 0.212    |
| eval/              |          |
|    coll_rew        | 100      |
|    mean_ep_length  | 120      |
|    mean_reward     | 31       |
| time/              |          |
|    total_timesteps | 686379   |
---------------------------------
New best mean reward!
Eval num_timesteps=687379, episode_reward=47.67 +/- 21.12
Episode length: 120.00 +/- 0.00
---------------------------------
| episode/           |          |
|    RML_rew         | 0.0382   |
|    high_speed_rew  | 0.302    |
| eval/              |          |
|    coll_rew        | 200      |
|    mean_ep_length  | 120      |
|    mean_reward     | 47.7     |
| time/              |          |
|    total_timesteps | 687379   |
---------------------------------
New best mean reward!
Eval num_timesteps=688379, ep

KeyboardInterrupt: 